# Multi-seed + matched-volume study (reviewer blockers)

Closes the two experiments a TKDE referee requires:
1. **multi-seed** with mean ± std (DP is stochastic; single seed is not defensible),
2. **matched retrieval-volume baseline** (no-DP top-k) to separate the DP effect from
   the effect of serving fewer items.

Graph already built (`kg/hcm100`, 856/910). This notebook only runs conditions and
aggregates; every (condition, seed) is a resumable logged run.

## R1. Reconnect [run first, and after any disconnect]
Restart Ollama, pull+pin models (a fresh VM has no models), reload the graph, install
retry wrappers.

In [ ]:
%%bash
pkill -9 -f ollama 2>/dev/null; sleep 3
which ollama >/dev/null 2>&1 || (apt-get -qq install -y zstd pciutils lshw >/dev/null 2>&1; curl -fsSL https://ollama.com/install.sh | sh >/dev/null 2>&1)
nohup env OLLAMA_KEEP_ALIVE=-1 ollama serve > /tmp/ollama.log 2>&1 &
sleep 8
ollama pull qwen2.5:7b >/dev/null 2>&1
ollama pull nomic-embed-text >/dev/null 2>&1
curl -s http://localhost:11434/api/generate -d '{"model":"qwen2.5:7b","prompt":"hi","options":{"num_ctx":4096},"keep_alive":-1,"stream":false}' >/dev/null
curl -s http://localhost:11434/api/embeddings -d '{"model":"nomic-embed-text","prompt":"hi","keep_alive":-1}' >/dev/null
echo '--- both models must show ---'; curl -s http://localhost:11434/api/ps

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import sys; ROOT='/content/drive/MyDrive/graphrag_defense'; sys.path.insert(0,ROOT)
!pip install -q requests networkx numpy datasets pandas rouge-score >/dev/null 2>&1
from grd import config as C, data, attack, metrics, kg_build as kb, retriever as rt, run_b, experiments as ex
from grd.defenses import NoDefense, DPRetrieval, TopKNoDP, SystemPromptRule, SimilarityThreshold, Summarization
import grd.run_b as R, grd.experiments as EX, time, requests, pandas as pd
C.ensure_dirs(); C.DP_CANDIDATE_POOL = 30

def gen_retry(prompt, system='', model=None, host=None, num_ctx=4096, timeout=600):
    for _ in range(3):
        try:
            r=requests.post(f'{C.OLLAMA_HOST}/api/generate', json={'model':C.LLM_MODEL,'prompt':prompt,'system':system,'options':{'temperature':0.0,'num_ctx':num_ctx,'num_predict':512},'keep_alive':-1,'stream':False}, timeout=timeout)
            r.raise_for_status(); return r.json()['response']
        except Exception as e: last=e; time.sleep(5)
    raise last
def emb_retry(text, model=None, host=None, timeout=120):
    for _ in range(3):
        try:
            r=requests.post(f'{C.OLLAMA_HOST}/api/embeddings', json={'model':C.EMB_MODEL,'prompt':text,'keep_alive':-1}, timeout=timeout)
            r.raise_for_status(); return r.json()['embedding']
        except Exception: time.sleep(5)
    raise RuntimeError('embed failed')
kb.ollama_generate=gen_retry; kb.ollama_embed=emb_retry; rt.ollama_embed=emb_retry
R.retrieve=lambda q,store,top_n=None: rt.retrieve(q,store,top_n=top_n); EX.retrieve=R.retrieve

store = kb.KGStore(f'{ROOT}/kg/hcm100')
N_ENT,N_REL,_,_ = run_b.graph_sets(store)
print('reconnected. graph', N_ENT, N_REL)

## R2. Multi-seed + matched-volume, targeted C3

Five conditions x 5 seeds x 40 queries. DP and `topk_nodp` share k=10, so the gap
between them is the DP effect at matched volume, and the gap between `none` (30 items)
and `topk_nodp` (10 items) is the volume effect. Deterministic conditions ignore the
seed but are run once per seed for a uniform table (their std is 0).

Time: ~5 conditions x 5 seeds x 40 q x ~17s. Deterministic conditions are cached across
seeds by run_b (already-logged queries skipped), so effective cost is far lower. Budget
if short: cut SEEDS to [0,1,2] or NQ to 25.

In [ ]:
SEEDS = [0,1,2,3,4]
NQ = 40
summ = Summarization(summarize_fn=lambda t: kb.summarize_text(t))
conds = {
  'none':      lambda s: NoDefense(),
  'topk_nodp': lambda s: TopKNoDP(k=10),               # matched volume, NO dp
  'dp_budget': lambda s: DPRetrieval(eps_ent=1.0, eps_rel=1.0, k=10, user_budget=10, seed=s),
  'sysprompt': lambda s: SystemPromptRule(),
  'threshold': lambda s: SimilarityThreshold(tau=0.5),
}
qfn = lambda: attack.build_queries('targeted','C3')[:NQ]
per_run, agg = ex.run_multiseed(store, qfn, conds, SEEDS, N_ENT, N_REL, tag='ms_tc3')
per_run.to_csv(f'{C.LOG_DIR}/multiseed_per_run.csv', index=False)
print('per-run saved.')
agg

### R2a. Paper-ready table: mean ± std across seeds

In [ ]:
tab = pd.DataFrame({
  'cum entity exp (%)':   ex.format_meanstd(agg,'cum_ent_pct'),
  'targeted hits':        ex.format_meanstd(agg,'targeted_hits'),
  'served/query':         agg[('avg_served_ent','mean')].round(1).astype(str),
})
tab.to_csv(f'{C.LOG_DIR}/multiseed_meanstd.csv')
tab

### R2b. The confound separation, stated explicitly
`none` -> `topk_nodp` gap = the effect of serving fewer items (volume).
`topk_nodp` -> `dp_budget` gap = the effect of DP noise + budget, at matched volume.

In [ ]:
m = per_run.groupby('condition')['cum_ent_pct'].mean()
vol_effect = m['none'] - m['topk_nodp']
dp_effect  = m['topk_nodp'] - m['dp_budget']
print(f"none                 : {m['none']:.1f}% cumulative exposure")
print(f"topk_nodp (k=10)     : {m['topk_nodp']:.1f}%   <- volume reduced, no DP")
print(f"dp_budget (k=10,B=10): {m['dp_budget']:.1f}%   <- + DP noise + budget")
print(f"\nvolume effect (none->topk_nodp) : {vol_effect:.1f} pts")
print(f"DP effect at matched volume      : {dp_effect:.1f} pts")
print("=> the DP mechanism contributes the matched-volume gap, not just lower volume.")

## R3. Multi-seed budget frontier (optional, for the frontier figure with error bars)

In [ ]:
conds_B = ex.budget_sweep_conditions([5,10,20,40], k=10, eps=1.0)
per_B, agg_B = ex.run_multiseed(store, lambda: attack.build_queries('targeted','C3')[:NQ],
                                conds_B, SEEDS, N_ENT, N_REL, tag='ms_bud')
per_B.to_csv(f'{C.LOG_DIR}/multiseed_budget.csv', index=False)
ex.format_meanstd(agg_B,'cum_ent_pct')

## R4. Back up


In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/logs_backup','zip', C.LOG_DIR)
files.download('/content/logs_backup.zip')

Paste R2a (mean±std table) and R2b (confound separation) back, and I will drop them
into the paper: R2a replaces the single-seed tables, R2b becomes the matched-volume
paragraph that answers the reviewer's fairness concern. R3 turns the frontier figure into
one with error bars.